## Gmail ophalen met SimpleGmail API

**Alle informatie over simpleGmail is te vinden op de github: https://github.com/jeremyephron/simplegmail**

Code is ook gebaseerd hierop

In [1]:
from simplegmail import Gmail
import pandas as pd
from langchain_groq import ChatGroq
import os
from pathlib import Path
import re
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
output_path = Path.cwd().parent.parent / "data" / "e-mails_fietsvergoeding" / "emails.csv"

# maak aan indien nodig
output_path.parent.mkdir(parents=True, exist_ok=True)

In [3]:
# duidt dep2526g09@@gmail.com aan (momenteel manueel)
gmail = Gmail()

In [ ]:
query_params = {
    "newer_than": (1,"day"), # gisteren
    "query": 'subject:"Aanvraag fietsvergoeding"' # pak enkel de emails met Aanvraag fietsvergoeding in de subject
}

In [5]:
messages = gmail.get_messages(query=query_params) # we nemen enkel de messgaes die nieuwer zijn dan gisteren, als we een dag missen moeten we de params aanpassen

In [6]:
for message in messages:
    print(message.id)

19dcfc6940f3bc05
19d3e204bdd2ed10
19d3d9ded1beec12
19d1a48f3693579c
19cd19fda441d3d2
19cd19bcc4965ca7
19cd1980813a1168
19c7091523849a71
19c7090fa49f1ad4
19c70908999f6420


In [7]:
# voor nu checken we of de csv al bestaat, indien niet gaan we een Dataframe maken, indien wel lezen we deze in
# later gaan we gwn checken of er al data is in de database, als dit is dan gaan we gwn fetchen en anders Dataframe maken zoals nu
try:
    df = pd.read_csv(output_path,sep=";")
except:
    df = pd.DataFrame(columns=['Id','DateKey', 'TimeKey', 'From', 'Subject', 'Message'])

In [8]:
for message in messages:
    # check op sender en of de id al gezien is
    if message.sender == "Sabine De Vreese <sabine.devreese@hogent.be>" and message.id not in df['Id'].values:
        dt = pd.to_datetime(message.date)
        
        # We voegen de hele rij in één keer toe als een lijst
        df.loc[len(df)] = [
            message.id,
            dt.strftime("%Y%m%d"), #datekey YYYYMMDD
            dt.strftime("%H:%M:%S"), #timekey HH:MM:SS
            message.sender, 
            message.subject, 
            message.plain 
        ]

In [9]:
df

,Id,DateKey,TimeKey,From,Subject,Message
0,19c7091523849a71,20260218,12:44:55,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
1,19c7090fa49f1ad4,20260218,12:44:31,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
2,19c70908999f6420,20260218,12:44:02,Sabine De Vreese <sabine.devreese@hogent.be>,FW: Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
3,19cd19fda441d3d2,20260309,09:03:52,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
4,19cd19bcc4965ca7,20260309,08:59:31,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,\r\nBeste\r\n\r\nBij deze mijn aanvraag voor e...
5,19cd1980813a1168,20260309,08:55:22,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,\r\nBeste\r\n\r\nBij deze mijn aanvraag voor e...
6,19d3e204bdd2ed10,20260330,11:43:12,Sabine De Vreese <sabine.devreese@hogent.be>,Re: Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
7,19d3d9ded1beec12,20260330,09:20:48,Sabine De Vreese <sabine.devreese@hogent.be>,Re: Aanvraag fietsvergoeding,\r\nBeste\r\n\r\nBij deze mijn aanvraag voor e...
8,19d1a48f3693579c,20260323,11:41:19,Sabine De Vreese <sabine.devreese@hogent.be>,FW: Aanvraag fietsvergoeding,\r\nBeste\r\n\r\nBij deze mijn aanvraag voor e...


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Id       9 non-null      str  
 1   DateKey  9 non-null      int64
 2   TimeKey  9 non-null      str  
 3   From     9 non-null      str  
 4   Subject  9 non-null      str  
 5   Message  9 non-null      str  
dtypes: int64(1), str(5)
memory usage: 7.2 KB


In [11]:
df.head()

,Id,DateKey,TimeKey,From,Subject,Message
0,19c7091523849a71,20260218,12:44:55,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
1,19c7090fa49f1ad4,20260218,12:44:31,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
2,19c70908999f6420,20260218,12:44:02,Sabine De Vreese <sabine.devreese@hogent.be>,FW: Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
3,19cd19fda441d3d2,20260309,09:03:52,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
4,19cd19bcc4965ca7,20260309,08:59:31,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,\r\nBeste\r\n\r\nBij deze mijn aanvraag voor e...


In [12]:
# Stap 1: Zet alles om naar getallen, maak van fouten NaN
df['DateKey'] = pd.to_numeric(df['DateKey'], errors='coerce')

# Stap 2: Nu kun je het veilig naar Int64 zetten (die gaat goed om met NaN)
df['DateKey'] = df['DateKey'].astype("Int64")

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Id       9 non-null      str  
 1   DateKey  9 non-null      Int64
 2   TimeKey  9 non-null      str  
 3   From     9 non-null      str  
 4   Subject  9 non-null      str  
 5   Message  9 non-null      str  
dtypes: Int64(1), str(5)
memory usage: 7.2 KB


In [14]:
df

,Id,DateKey,TimeKey,From,Subject,Message
0,19c7091523849a71,20260218,12:44:55,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
1,19c7090fa49f1ad4,20260218,12:44:31,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
2,19c70908999f6420,20260218,12:44:02,Sabine De Vreese <sabine.devreese@hogent.be>,FW: Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
3,19cd19fda441d3d2,20260309,09:03:52,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
4,19cd19bcc4965ca7,20260309,08:59:31,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,\r\nBeste\r\n\r\nBij deze mijn aanvraag voor e...
5,19cd1980813a1168,20260309,08:55:22,Sabine De Vreese <sabine.devreese@hogent.be>,Aanvraag fietsvergoeding,\r\nBeste\r\n\r\nBij deze mijn aanvraag voor e...
6,19d3e204bdd2ed10,20260330,11:43:12,Sabine De Vreese <sabine.devreese@hogent.be>,Re: Aanvraag fietsvergoeding,Beste\r\n\r\nBij deze mijn aanvraag voor een f...
7,19d3d9ded1beec12,20260330,09:20:48,Sabine De Vreese <sabine.devreese@hogent.be>,Re: Aanvraag fietsvergoeding,\r\nBeste\r\n\r\nBij deze mijn aanvraag voor e...
8,19d1a48f3693579c,20260323,11:41:19,Sabine De Vreese <sabine.devreese@hogent.be>,FW: Aanvraag fietsvergoeding,\r\nBeste\r\n\r\nBij deze mijn aanvraag voor e...


In [15]:
df.to_csv(output_path, index=False, sep=';', encoding='utf-8')

In [16]:
GROQ_API_KEY = os.getenv("groq_API")

In [17]:
llm = ChatGroq(
    temperature=0, 
    model_name="llama-3.3-70b-versatile", 
    groq_api_key=GROQ_API_KEY)

In [18]:
def ai(mail):

    prompt = f"""

    Je bent een email analyst. Je taak is om een e-mails rejecten of niet op basis van REGELS die worden opgelijst hieronder.

    Te analyseren e-mail:
    -----------------------
    {mail}
    -----------------------


    STRIKTE REGELS VOORDAT EEN EMAIL WORDT GE ACCEPT:
    1. Kijk naar de 'Periode' in de mail (bijv. 09/2025). Dit bepaalt de maand.
    2. Scan de tekst op specifieke datums (bijv. "dinsdag 2 september 2025").
    3. De afstand opgegeven in de mail moet realistisch zijn om te fietsen, niet groter dan 30km
    4. Er mogen geen dubbele dagen instaan

    ANTWOORD ENKEL met het woord 'REJECTED' indien rejected of 'ACCEPTED' indien accepted.:
    """
    
    return llm.invoke(prompt).content.strip()

### Regex Cheat Sheet voor Data Extractie

| Teken | Matcht | Voorbeeld |
| :--- | :--- | :--- |
| `.` | Elk teken (behalve nieuwe regel) | `a.c` -> abc, a1c, a!c |
| `\d` | Elk cijfer (0-9) | `\d\d` -> 15, 99 |
| `\D` | Alles BEHALVE een cijfer | `\D` -> A, !, spatie |
| `\w` | Letter, cijfer of underscore | `\w` -> a, B, 5, _ |
| `\s` | Whitespace (spatie, tab, enter) | `\s` -> " " |
| `\\` | Een letterlijke backslash | `\\` -> \ |
| `\.` | Een letterlijke punt | `\.` -> . |
| `*` | 0 of meer keer | `ab*` -> a, ab, abbb |
| `+` | 1 of meer keer | `ab+` -> ab, abbb (niet: a) |
| `?` | 0 of 1 keer (optioneel) | `€?` -> matcht € of niets |
| `{n}` | Exact n keer | `\d{4}` -> 2025 |
| `{n,}` | Minimaal n keer | `\d{2,}` -> 12, 123, 1234... |
| `{n,m}` | Tussen n en m keer | `\d{2,4}` -> 12, 123, 1234 |
| `(...)` | Capture Group | Alles hierbinnen kun je ophalen met `.group(1)` |
| `[abc]` | Character Set | Matcht 'a', 'b' of 'c' |
| `[a-z]` | Range | Elke kleine letter van a t/m z |
| `[^0-9]` | Negatie | Alles BEHALVE een cijfer |
| `(x\|y)` | Of-functie | Matcht 'x' of 'y' |
| `^` | Begin van regel | `^Start` matcht alleen aan het begin |
| `$` | Einde van regel | `Einde$` matcht alleen aan het einde |
| `\b` | Woordgrens | `\bkm\b` matcht "km" maar niet "kilometers" |

In [19]:
def extract_data(text):
    # We controleren eerst of de mail geaccepteerd wordt
    if ai(text) == 'ACCEPTED':

        
        pid = re.search(r"PersoneelsID:\s*(\d+)", text).group(1)
        entiteit = re.search(r"Gekozen entiteit:.*-\s*(.*)", text).group(1).strip()
        periode = re.search(r"Periode:\s*(\d{2}/\d{4})", text).group(1)
        
        # Afstand (ook direct met group(1))
        afstand_str = re.search(r"Afstand:\s*(\d+(?:,\d+)?)", text).group(1)
        afstand = float(afstand_str.replace(",", "."))
        
        # Totalen
        rit_enkel = int(re.search(r"Totaal aantal dagen enkele rit:\s*(\d+)", text).group(1))
        rit_dubbel = int(re.search(r"Totaal aantal dagen heen-en-terug:\s*(\d+)", text).group(1))
        
        # Vertrekplaats
        departure_place = re.search(r"Vertrekplaats:\s*(.*)", text).group(1).strip()
          
        # Berekeningen
        totaal_km = afstand * ((rit_dubbel * 2) + rit_enkel)
        
        # Postcode verkrijgen met AI (jouw specifieke implementatie)
        postalCode = llm.invoke(input=f"Geef de Belgische postcode van de plaats {departure_place}. GEEF ALLEEN ALS OUTPUT DE POSTCODE").content.strip()
        
        # Dagen logica
        dagen = [0.0] * 31
        dag_regels = re.findall(r"(\w+)\s+(\d+)\s+(\w+)\s+(\d{4}):\s+(.*)", text)
        
        # Controle of de lijst met regels overeenkomt met de opgegeven totalen
        if rit_dubbel + rit_enkel == len(dag_regels):
            for _, dag_nr, _, _, type_rit in dag_regels:
                idx = int(dag_nr) - 1 # Index is dagnummer - 1
                
                if 0 <= idx < 31:
                    if "heen en terug" in type_rit.lower():
                        dagen[idx] = afstand * 2
                    else:
                        dagen[idx] = afstand

        return [pid, entiteit, departure_place, postalCode, totaal_km, periode] + dagen
    
    return None

In [20]:
df = pd.read_csv("../../data/e-mails_fietsvergoeding/emails.csv", sep=';')

In [21]:
def analyser(mails): 
    # Kolommen altijd definiëren, ongeacht of mails string of lijst is
    kolommen = ["StaffID", "Entity", "Campus", "PostalCode", "DistanceKM", "Period"] + [f"dag {i}" for i in range(1, 32)] 
    
    # Als mails slechts één string is, zet om naar lijst
    if isinstance(mails, str): 
        mails = [mails] 

    data = [] 
    for mail in mails: 
        resultaat = extract_data(mail) 
        if resultaat is not None: 
            data.append(resultaat)  # gebruik resultaat, niet opnieuw extract_data(mail)
    
    # return pas na de loop
    return pd.DataFrame(data, columns=kolommen)

In [22]:
# Gebruik nu:
resultaat_df = analyser(df['Message'].tolist()) # Stuurt de hele kolom

In [23]:
resultaat_df

,StaffID,Entity,Campus,PostalCode,DistanceKM,Period,dag 1,dag 2,dag 3,dag 4,...,dag 22,dag 23,dag 24,dag 25,dag 26,dag 27,dag 28,dag 29,dag 30,dag 31
0,1000,DIT,Deinze,9800,261.0,09/2025,0.0,29.0,0.0,29.0,...,0.0,29.0,29.0,29.0,0.0,0.0,0.0,0.0,29.0,0.0
1,1000,DIT,Deinze,9800,348.0,03/2025,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,29.0,29.0,29.0,0.0,0.0,0.0
2,1000,DIT,Deinze,9800,348.0,11/2025,0.0,0.0,29.0,29.0,...,0.0,0.0,0.0,29.0,29.0,29.0,0.0,0.0,0.0,0.0
3,437,DIT,Zingem,9790,234.0,09/2025,0.0,13.0,0.0,13.0,...,0.0,13.0,13.0,26.0,0.0,0.0,0.0,0.0,26.0,0.0
4,442,DIT,Aalter,9910,261.0,09/2025,0.0,29.0,0.0,29.0,...,0.0,29.0,29.0,29.0,0.0,0.0,0.0,0.0,29.0,0.0
5,2000,DIT,Sint-Martens-Latem,9830,40.0,03/2026,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,16.0,0.0,0.0,8.0,16.0
6,1000,DIT,Deinze,9800,45.0,02/2026,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,30.0,15.0,0.0,0.0,0.0,0.0
7,1000,DIT,Deinze,9800,45.0,02/2026,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,30.0,15.0,0.0,0.0,0.0,0.0


In [24]:
resultaat_df['Period'] = pd.to_datetime(resultaat_df['Period'], format='%m/%Y')

# 2. Formatteer naar het gewenste formaat 'sep/25'
# %b is de afgekorte maandnaam, %y is het korte jaartal
# .str.lower() zorgt dat het 'sep' wordt in plaats van 'Sep'
resultaat_df['Period'] = resultaat_df['Period'].dt.strftime('%b/%y').str.lower()

In [25]:
resultaat_df.head()

,StaffID,Entity,Campus,PostalCode,DistanceKM,Period,dag 1,dag 2,dag 3,dag 4,...,dag 22,dag 23,dag 24,dag 25,dag 26,dag 27,dag 28,dag 29,dag 30,dag 31
0,1000,DIT,Deinze,9800,261.0,sep/25,0.0,29.0,0.0,29.0,...,0.0,29.0,29.0,29.0,0.0,0.0,0.0,0.0,29.0,0.0
1,1000,DIT,Deinze,9800,348.0,mar/25,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,29.0,29.0,29.0,0.0,0.0,0.0
2,1000,DIT,Deinze,9800,348.0,nov/25,0.0,0.0,29.0,29.0,...,0.0,0.0,0.0,29.0,29.0,29.0,0.0,0.0,0.0,0.0
3,437,DIT,Zingem,9790,234.0,sep/25,0.0,13.0,0.0,13.0,...,0.0,13.0,13.0,26.0,0.0,0.0,0.0,0.0,26.0,0.0
4,442,DIT,Aalter,9910,261.0,sep/25,0.0,29.0,0.0,29.0,...,0.0,29.0,29.0,29.0,0.0,0.0,0.0,0.0,29.0,0.0


In [26]:
output_path = output_path = Path.cwd().parent.parent / "data" / "e-mails_fietsvergoeding" / "emails_final.csv"
resultaat_df.to_csv(output_path, index=False, sep=';', encoding='utf-8')